# 09 — Build a bulk cross-run analysis

This notebook runs the independent post-processing pipeline directly over a read-only archive of completed normal-pipeline flightline directories. Arbitrary outer batch/storage names are ignored scientifically. The pipeline streams aligned ENVI windows into compact sufficient statistics, then creates canonical catalogs, balanced synthetic MicaSense-to-Landsat summaries, and leave-one-site-out validation without copying source pixels.

## 1. Configure the source tree and output directory

`input_mode="auto"` prefers canonical inner flightline directories such as `NIWO_a01/NEON_D13_NIWO_DP1_L012-1_20230815_directional_reflectance/` and falls back to prebuilt merged Parquets. The output and scratch paths must be outside the read-only input tree. Start with `preflight_only = True`; it reads only names, file metadata, ENVI headers, and QA JSON, and creates no pixel cache.

In [ ]:
from pathlib import Path
from pprint import pprint

import duckdb

from spectralbridge import run_bulk_pipeline
from spectralbridge.bulk import SpectralLibraryPlotConfig

RUN = False
input_root = Path("/home/jovyan/data-store/Aug_2026_Processed_Flightlines")
output_dir = Path("/home/jovyan/data-store/Aug_2026_Bulk_Analysis")
temp_directory = Path("/home/jovyan/work/spectralbridge_bulk_scratch")
input_mode = "auto"
input_kind = "full"
memory_limit = "175GB"
threads = 16
extraction_workers = 1  # conservative: storage bandwidth is often limiting
extraction_chunk_size = 2048
diagnostic_sample_size = 0  # optional global bound, never all pixels
preflight_only = True
materialize_observations = False

# Optional existing merged polygon library; it is read in place, not copied.
spectral_library = None  # Path("/data/library/polygons_merged_pixel_extraction.parquet")
make_summary_plots = False
make_full_spectral_reports = False
spectral_plot_config = SpectralLibraryPlotConfig(
    spectral_stage="corr",
    species_sort="count_desc",
    panels_per_page=4,
    trace_batch_size=2_000,
    max_traces_per_group=None,  # all valid traces by default
)

## 2. Build or reuse the collection

The pipeline recovers canonical flightline identity from the inner NEON directory, never from outer distributed-compute folders. After reviewing preflight, set `preflight_only = False`. Target-sensor ENVI products are read in bounded aligned windows; correction and convolution are not recomputed. Successful per-flightline statistics checkpoints are reused independently.

In [ ]:
result = None
if RUN:
    result = run_bulk_pipeline(
        input_root,
        output_dir,
        input_mode=input_mode,
        input_kind=input_kind,
        memory_limit=memory_limit,
        threads=threads,
        extraction_workers=extraction_workers,
        extraction_chunk_size=extraction_chunk_size,
        diagnostic_sample_size=diagnostic_sample_size,
        temp_directory=temp_directory,
        preflight_only=preflight_only,
        materialize_observations=materialize_observations,
        spectral_library=spectral_library,
        make_summary_plots=make_summary_plots,
        make_full_spectral_reports=make_full_spectral_reports,
        spectral_library_config=spectral_plot_config,
    )
    pprint(result)
else:
    print("Edit the paths and set RUN = True to build the bulk collection.")

## 3. Inspect catalogs and analyses

The database stores canonical flightline and source-product catalogs, compact sufficient statistics, models, and provenance. Completed-output analysis does not use `bulk_observations`; accepted original merged Parquets remain a read-in-place compatibility mode. Pixel-level materialization is a separate explicit dataset-build operation.

In [ ]:
if RUN and result is not None:
    with duckdb.connect(result["database"], read_only=True) as con:
        source_summary = con.execute(
            "SELECT status, COUNT(*) AS flightlines, SUM(row_count) AS rows "
            "FROM flightlines GROUP BY status ORDER BY status"
        ).df()
        census = con.execute("SELECT * FROM dataset_census_summary").df()
        coefficients = None
        if not preflight_only:
            coefficients = con.execute(
                "SELECT landsat_sensor, band_index, analysis_level, slope, "
                "intercept, r2, flightline_count, site_count "
                "FROM candidate_translation_coefficients "
                "ORDER BY landsat_sensor, band_index, analysis_level"
            ).df()
    display(source_summary)
    display(census)
    if coefficients is not None:
        display(coefficients)

## 4. Inspect optional spectral-library products

Summary plotting writes species-median and observation-count PDFs. Full reporting adds the multipage low-alpha species, quantile, hierarchy, site, and flightline reports. The default renders every complete finite trace in bounded rasterized batches; only an explicit `max_traces_per_group` enables deterministic sampling.

In [ ]:
if RUN and result is not None and result.get("spectral_library"):
    spectral_result = result["spectral_library"]
    pprint(spectral_result["schema"])
    pprint(spectral_result["reports"])
    with duckdb.connect(result["database"], read_only=True) as con:
        display(
            con.execute(
                "SELECT * FROM spectral_library_species_summary "
                "ORDER BY valid_spectrum_count DESC, species"
            ).df()
        )

## 5. Interpret the result

Each equation is `Landsat = slope × MicaSense + intercept`. Pixel-pooled results allow large flightlines to contribute more observations; flightline- and site-balanced results give each replicate equal total weight. These same-source synthetic relationships are separate from fixed percentage brightness adjustment and are not empirical field calibration.

In [ ]:
for artifact in sorted(path for path in output_dir.rglob("*") if path.is_file()):
    print(artifact.relative_to(output_dir), artifact.stat().st_size)